# PGA-UNet vs U-Net 2D — Test & So sánh (IMG_SIZE=512)

| Phần | Nội dung |
|---|---|
| **Setup** | Clone 2 repo riêng biệt, download dataset + checkpoints |
| **Phần 1** | PGA-UNet: test 3 prompt modes + visualization |
| **Phần 2** | U-Net 2D: test + visualization |
| **Tổng hợp** | Bảng so sánh 6 metrics + biểu đồ cột |

Chạy tuần tự: **Setup → Phần 1 → Phần 2 → Tổng hợp**

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1 — SETUP (PGA-UNet + U-Net 2D)
# ══════════════════════════════════════════════════════
import os, sys, gdown, torch

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO = 'https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git'

# ── Repo: PGA (branch TN_B_ON) → pga-repo ───────────────────────────
if not os.path.exists(f'{BASE}/pga-repo'):
    os.system(f'git clone -q -b TN_B_ON {REPO} {BASE}/pga-repo')
print('  ✅ pga-repo')

# ── Repo: U-Net (default branch) → unet-repo ────────────────────────
if not os.path.exists(f'{BASE}/unet-repo'):
    os.system(f'git clone -q {REPO} {BASE}/unet-repo')
print('  ✅ unet-repo')

# ── Dataset ───────────────────────────────────────────────────────────
DS_ZIP = f'{BASE}/dataset_BTXRD.zip'
if not os.path.exists(DS_ZIP):
    gdown.download('https://drive.google.com/uc?id=1X1SY8T63pdBPIdrv_3P0gKVwoLxCa5sW',
                   DS_ZIP, quiet=False)
for repo in ['pga-repo', 'unet-repo']:
    if not os.path.exists(f'{BASE}/{repo}/dataset_BTXRD'):
        os.system(f'unzip -oq {DS_ZIP} -d {BASE}/{repo}/')
print('  ✅ Dataset ready (cả 2 repo)')

# ── Checkpoints ───────────────────────────────────────────────────────
for repo, fn, drv_id in [
    ('pga-repo',  'pga_unet_expB_best.pth', '1sSoQ8SObteWETuKSsGGhBASPtXQz9JbY'),
    ('unet-repo', 'unet_best.pth',           '1hKyyw8WvN6WRyzjDE50upWI0NIDJbSMu'),
]:
    os.makedirs(f'{BASE}/{repo}/checkpoints', exist_ok=True)
    p = f'{BASE}/{repo}/checkpoints/{fn}'
    if not os.path.exists(p):
        gdown.download(f'https://drive.google.com/uc?id={drv_id}', p, quiet=False)
    print(f'  ✅ {fn}  {os.path.getsize(p)//1024} KB')

os.system('pip install -q tqdm opencv-python matplotlib scipy gdown')
print(f'\n✅ Setup DONE  |  device={"cuda" if torch.cuda.is_available() else "cpu"}')


---
## Phần 1 — PGA-UNet (IMG_SIZE=512, 3 prompt modes)
---

In [ ]:
# ══════════════════════════════════════════════════════
# PHẦN 1 — PGA-UNet Test (IMG_SIZE=512, 3 prompt modes)
# ══════════════════════════════════════════════════════
import os, csv, sys
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt
from collections import OrderedDict

BASE     = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT = f'{BASE}/pga-repo'

# ── Clear module cache rồi load PGA modules ──────────────────────────
for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset','models','prompt_unet')): del sys.modules[_k]
if PGA_ROOT not in sys.path: sys.path.insert(0, PGA_ROOT)
else: sys.path.remove(PGA_ROOT); sys.path.insert(0, PGA_ROOT)

from dataset import BTXRD_Dataset
from models.networks.prompt_unet_2D import PGA_UNet

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 512
IMG_DIR  = f'{PGA_ROOT}/dataset_BTXRD/test/images'
JSON_DIR = f'{PGA_ROOT}/dataset_BTXRD/test/annotations'
CKPT_PATH= f'{PGA_ROOT}/checkpoints/pga_unet_expB_best.pth'
os.makedirs(f'{PGA_ROOT}/results', exist_ok=True)

model = PGA_UNet(in_channels=1, n_classes=1, use_encoder_prompt=True).to(DEVICE)
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True))
model.eval()
print(f'✅ PGA-UNet loaded  [{DEVICE}]')

def calc_hd95(pred, gt):
    p, g = pred.astype(bool), gt.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or  not g.any(): return float(IMG_SIZE)
    pe = p ^ binary_erosion(p); ge = g ^ binary_erosion(g)
    d1 = distance_transform_edt(~ge)[pe]; d2 = distance_transform_edt(~pe)[ge]
    return float(IMG_SIZE) if not len(d1) or not len(d2) else float(max(np.percentile(d1,95),np.percentile(d2,95)))

def calc_metrics_img(prob_np, gt_np):
    pm=(prob_np>0.5).astype(np.float32); gm=(gt_np>0.5).astype(np.float32); eps=1e-6
    tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
    hd95=calc_hd95(pm,gm)
    if gm.sum()==0 or pm.sum()==0: cbl=0.0
    else:
        ys,xs=np.where(gm>0.5); yp,xp=np.where(pm>0.5)
        d=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+eps
        cbl=float(np.clip(1.-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/d,0,1))
    return dict(dice=float((2*tp+eps)/(2*tp+fp+fn+eps)), iou=float((tp+eps)/(tp+fp+fn+eps)),
                precision=float((tp+eps)/(tp+fp+eps)),   recall=float((tp+eps)/(tp+fn+eps)),
                hd95=hd95, cbl=cbl)

KEYS  = ['dice','iou','precision','recall','hd95','cbl']
HDRS  = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓','CBL↑']
MODES = ['zoom_out','shift','mixed_7_3']
all_image_records = {}
pga_csv_rows = []
pga_results  = {}   # dùng ở cell tổng hợp

for mode in MODES:
    ds     = BTXRD_Dataset(IMG_DIR, JSON_DIR, img_size=IMG_SIZE, is_train=False, prompt_mode=mode)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    groups = OrderedDict()
    for i, (img_t, mask_t, prompt_t) in enumerate(tqdm(loader, desc=f'[{mode}]')):
        img_name, _ = ds.all_samples[i]
        gt_np=mask_t[0,0].numpy(); prompt_np=prompt_t[0,0].numpy()
        with torch.no_grad():
            prob=torch.sigmoid(model(img_t.to(DEVICE),prompt_t.to(DEVICE)))[0,0].cpu().numpy()
        if img_name not in groups:
            groups[img_name]=dict(img=img_t[0,0].numpy(),gt_union=gt_np.copy(),
                                  prob_max=prob.copy(),prompts=[prompt_np])
        else:
            g=groups[img_name]
            np.maximum(g['gt_union'],gt_np,out=g['gt_union'])
            np.maximum(g['prob_max'],prob, out=g['prob_max'])
            g['prompts'].append(prompt_np)
    img_recs=[]
    for img_name in sorted(groups.keys()):
        g=groups[img_name]; m=calc_metrics_img(g['prob_max'],g['gt_union'])
        img_recs.append(dict(img_name=img_name,img=g['img'],gt=g['gt_union'],
                             prob=g['prob_max'],prompts=g['prompts'],
                             n_samples=len(g['prompts']),**m))
    all_image_records[mode]=img_recs
    m_avg={k:np.mean([r[k] for r in img_recs]) for k in KEYS}
    pga_results[mode]=m_avg
    n_imgs=len(img_recs); n_samp=sum(r['n_samples'] for r in img_recs)
    pga_csv_rows.append([mode]+[f'{m_avg[k]:.4f}' for k in KEYS]+[str(n_imgs),str(n_samp)])

bar='='*82
print(f'\n{bar}\n  PGA-UNet — Image-level metrics (GT union+max-merge)  IMG_SIZE={IMG_SIZE}\n{bar}')
print(f"  {'Mode':<16}"+''.join(f'{h:>8}' for h in HDRS)+f"  {'N_img':>6}  {'N_smp':>6}")
print(f"  {'-'*78}")
for row in pga_csv_rows:
    print(f"  {row[0]:<16}"+''.join(f'{row[i+1]:>8}' for i in range(len(KEYS)))+f"  {row[-2]:>6}  {row[-1]:>6}")
print(bar)

with open(f'{PGA_ROOT}/results/pga_unet2d_test_results.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['mode']+KEYS+['N_img','N_samples']); w.writerows(pga_csv_rows)
print(f'\n✅ CSV: {PGA_ROOT}/results/pga_unet2d_test_results.csv')


### Visualization PGA-UNet — ảnh có ≥2 GT polygon

In [ ]:
# ── Visualization PGA-UNet: ≥10 ảnh có ≥2 GT polygon (zoom_out) ────
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert 'all_image_records' in dir(), '❌ Chạy cell PGA Test trước'
N_MIN=10; MIN_GT=2
recs=[r for r in all_image_records['zoom_out'] if r['n_samples']>=MIN_GT]
assert len(recs)>=N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT'
N_SHOW=len(recs)

fig,axes=plt.subplots(N_SHOW,5,figsize=(20,4*N_SHOW))
if N_SHOW==1: axes=axes[np.newaxis,:]
fig.suptitle(f'PGA-UNet — {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (zoom_out, image-level)',fontsize=13,y=1.001)
for ax,ct in zip(axes[0],['Ảnh gốc','Ảnh + Prompts','Dự đoán (merged)','Ground Truth','TP/FP/FN']):
    ax.set_title(ct,fontsize=10,fontweight='bold')

for count,rec in enumerate(recs):
    img_np=rec['img']*0.5+0.5
    gt_np=(rec['gt']>0.5).astype(float); pred_np=(rec['prob']>0.5).astype(float)
    pm_merged=np.max(np.stack(rec['prompts'],axis=0),axis=0)
    tp=(pred_np*gt_np).sum(); fp=(pred_np*(1-gt_np)).sum(); fn=((1-pred_np)*gt_np).sum(); e=1e-6
    dice=float((2*tp+e)/(2*tp+fp+fn+e)); iou=float((tp+e)/(tp+fp+fn+e))
    pre=float((tp+e)/(tp+fp+e));          rec_=float((tp+e)/(tp+fn+e))
    row=axes[count]; bg=np.stack([img_np]*3,axis=-1)

    row[0].imshow(img_np,cmap='gray',vmin=0,vmax=1)
    row[0].set_ylabel(f'#{count+1} [{rec["n_samples"]}p]\nDice={dice:.3f}',fontsize=8)
    row[1].imshow(img_np,cmap='gray',vmin=0,vmax=1)
    row[1].imshow(pm_merged,cmap='hot',alpha=0.4,vmin=0,vmax=1)
    row[1].set_title(rec['img_name'],fontsize=6,pad=2)
    pr_ov=bg.copy(); pr_ov[...,0]=np.clip(pr_ov[...,0]+pred_np*0.55,0,1); pr_ov[...,1]=np.clip(pr_ov[...,1]-pred_np*0.2,0,1)
    row[2].imshow(pr_ov); row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}',fontsize=7,color='darkred',pad=2)
    gt_ov=bg.copy(); gt_ov[...,1]=np.clip(gt_ov[...,1]+gt_np*0.55,0,1); gt_ov[...,0]=np.clip(gt_ov[...,0]-gt_np*0.2,0,1)
    row[3].imshow(gt_ov)
    inter=bg.copy()*0.25
    inter[...,1]=np.clip(inter[...,1]+pred_np*gt_np*0.9,0,1)
    inter[...,0]=np.clip(inter[...,0]+pred_np*(1-gt_np)*1.0,0,1)
    inter[...,2]=np.clip(inter[...,2]+(1-pred_np)*gt_np*1.0,0,1)
    row[4].imshow(inter); row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}',fontsize=7,color='saddlebrown',pad=2)
    for ax in row: ax.axis('off')

fig.legend(handles=[Patch(facecolor='green',label='TP'),Patch(facecolor='red',label='FP'),Patch(facecolor='blue',label='FN')],
           loc='lower center',ncol=3,fontsize=9,bbox_to_anchor=(0.5,-0.005))
plt.tight_layout(); _ipy_display(fig); plt.close(fig)


### Visualization PGA-UNet — ảnh có ≥2 GT polygon (shift)

In [ ]:
# ── Visualization PGA-UNet: ≥10 ảnh có ≥2 GT polygon (shift) ────
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from IPython.display import display as _ipy_display

assert 'all_image_records' in dir(), '❌ Chạy cell PGA Test trước'
N_MIN=10; MIN_GT=2
recs=[r for r in all_image_records['shift'] if r['n_samples']>=MIN_GT]
assert len(recs)>=N_MIN, f'Chỉ có {len(recs)} ảnh có ≥{MIN_GT} GT'
N_SHOW=len(recs)

fig,axes=plt.subplots(N_SHOW,5,figsize=(20,4*N_SHOW))
if N_SHOW==1: axes=axes[np.newaxis,:]
fig.suptitle(f'PGA-UNet — {N_SHOW} ảnh có ≥{MIN_GT} GT polygon (shift, image-level)',fontsize=13,y=1.001)
for ax,ct in zip(axes[0],['Ảnh gốc','Ảnh + Prompts','Dự đoán (merged)','Ground Truth','TP/FP/FN']):
    ax.set_title(ct,fontsize=10,fontweight='bold')

for count,rec in enumerate(recs):
    img_np=rec['img']*0.5+0.5
    gt_np=(rec['gt']>0.5).astype(float); pred_np=(rec['prob']>0.5).astype(float)
    pm_merged=np.max(np.stack(rec['prompts'],axis=0),axis=0)
    tp=(pred_np*gt_np).sum(); fp=(pred_np*(1-gt_np)).sum(); fn=((1-pred_np)*gt_np).sum(); e=1e-6
    dice=float((2*tp+e)/(2*tp+fp+fn+e)); iou=float((tp+e)/(tp+fp+fn+e))
    pre=float((tp+e)/(tp+fp+e));          rec_=float((tp+e)/(tp+fn+e))
    row=axes[count]; bg=np.stack([img_np]*3,axis=-1)

    row[0].imshow(img_np,cmap='gray',vmin=0,vmax=1)
    row[0].set_ylabel(f'#{count+1} [{rec["n_samples"]}p]\nDice={dice:.3f}',fontsize=8)
    row[1].imshow(img_np,cmap='gray',vmin=0,vmax=1)
    row[1].imshow(pm_merged,cmap='hot',alpha=0.4,vmin=0,vmax=1)
    row[1].set_title(rec['img_name'],fontsize=6,pad=2)
    pr_ov=bg.copy(); pr_ov[...,0]=np.clip(pr_ov[...,0]+pred_np*0.55,0,1); pr_ov[...,1]=np.clip(pr_ov[...,1]-pred_np*0.2,0,1)
    row[2].imshow(pr_ov); row[2].set_title(f'Dice={dice:.3f}  IoU={iou:.3f}',fontsize=7,color='darkred',pad=2)
    gt_ov=bg.copy(); gt_ov[...,1]=np.clip(gt_ov[...,1]+gt_np*0.55,0,1); gt_ov[...,0]=np.clip(gt_ov[...,0]-gt_np*0.2,0,1)
    row[3].imshow(gt_ov)
    inter=bg.copy()*0.25
    inter[...,1]=np.clip(inter[...,1]+pred_np*gt_np*0.9,0,1)
    inter[...,0]=np.clip(inter[...,0]+pred_np*(1-gt_np)*1.0,0,1)
    inter[...,2]=np.clip(inter[...,2]+(1-pred_np)*gt_np*1.0,0,1)
    row[4].imshow(inter); row[4].set_title(f'Pre={pre:.3f}  Rec={rec_:.3f}',fontsize=7,color='saddlebrown',pad=2)
    for ax in row: ax.axis('off')

fig.legend(handles=[Patch(facecolor='green',label='TP'),Patch(facecolor='red',label='FP'),Patch(facecolor='blue',label='FN')],
           loc='lower center',ncol=3,fontsize=9,bbox_to_anchor=(0.5,-0.005))
plt.tight_layout(); _ipy_display(fig); plt.close(fig)


---
## Phần 2 — U-Net 2D (IMG_SIZE=512, không có prompt)
---

In [ ]:
# ══════════════════════════════════════════════════════
# PHẦN 2 — U-Net 2D Test (IMG_SIZE=512)
# ══════════════════════════════════════════════════════
import os, csv, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt

BASE      = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
UNET_ROOT = f'{BASE}/unet-repo'

# ── Clear module cache rồi load UNet modules ─────────────────────────
for _k in list(sys.modules.keys()):
    if any(x in _k for x in ('dataset','models','unet')): del sys.modules[_k]
if UNET_ROOT not in sys.path: sys.path.insert(0, UNET_ROOT)
else: sys.path.remove(UNET_ROOT); sys.path.insert(0, UNET_ROOT)

from dataset import BTXRD_Dataset
from models.networks.unet_2D import unet_2D

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = f'{UNET_ROOT}/checkpoints/unet_best.pth'
IMG_SIZE   = 512

def calc_hd95_u(pred, gt):
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or  not gt.any(): return float(IMG_SIZE)
    pe=pred^binary_erosion(pred); ge=gt^binary_erosion(gt)
    d1=distance_transform_edt(~ge)[pe]; d2=distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1,95),np.percentile(d2,95)))

def calc_cbl_u(pred_bin, gt_bin):
    if gt_bin.sum()==0: return None
    ys,xs=np.where(gt_bin)
    gt_diag=np.sqrt((ys.max()-ys.min())**2+(xs.max()-xs.min())**2)+1e-6
    if pred_bin.sum()==0: return 0.0
    yp,xp=np.where(pred_bin)
    return float(np.clip(1.0-np.sqrt((xp.mean()-xs.mean())**2+(yp.mean()-ys.mean())**2)/gt_diag,0.0,1.0))

def get_centroid(m):
    if m.sum()==0: return None,None
    ys,xs=np.where(m); return float(xs.mean()),float(ys.mean())

model=unet_2D(in_channels=1,n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH,map_location=DEVICE,weights_only=True))
model.eval()
print(f'✅ U-Net 2D loaded  [{DEVICE}]  {os.path.getsize(MODEL_PATH)//1024} KB')

test_dataset=BTXRD_Dataset(image_dir=f'{UNET_ROOT}/dataset_BTXRD/test/images',
                            mask_dir =f'{UNET_ROOT}/dataset_BTXRD/test/masks',
                            img_size=IMG_SIZE, is_train=False)
test_dataset.images = sorted(test_dataset.images)
if hasattr(test_dataset, 'masks'):
    test_dataset.masks = sorted(test_dataset.masks)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Danh sách tên ảnh đúng thứ tự (sorted, khớp với DataLoader)
img_names_sorted = [os.path.basename(p) for p in test_dataset.images]

# Chọn ảnh để visualize: khớp với PGA (≥2 GT polygon) nếu có, không thì 10 ảnh đầu
if 'all_image_records' in dir() and 'zoom_out' in all_image_records:
    _pga_names = {r['img_name'] for r in all_image_records['zoom_out'] if r['n_samples'] >= 2}
    SHOW_INDEX  = {i for i, n in enumerate(img_names_sorted) if n in _pga_names}
    print(f'📌 Visualization: {len(SHOW_INDEX)} ảnh khớp với PGA (≥2 GT polygon)')
else:
    SHOW_INDEX = set(range(10))
    print(f'📌 Visualization: 10 ảnh đầu (all_image_records chưa có trong bộ nhớ)')

unet_all_dice,unet_all_iou,unet_all_pre,unet_all_rec,unet_all_hd95,unet_all_cbl=[],[],[],[],[],[]
smooth=1e-5

with torch.no_grad():
    for idx,(images,masks) in enumerate(test_loader):
        images,masks=images.to(DEVICE),masks.to(DEVICE)
        preds=(torch.sigmoid(model(images))>0.5).float()
        img_np=(images[0,0].cpu().numpy()+1)/2.0
        gm=masks[0,0].cpu().numpy(); pm=preds[0,0].cpu().numpy()
        tp=(pm*gm).sum(); fp=(pm*(1-gm)).sum(); fn=((1-pm)*gm).sum()
        dice=(2*tp+smooth)/(2*tp+fp+fn+smooth); iou=(tp+smooth)/(tp+fp+fn+smooth)
        pre=(tp+smooth)/(tp+fp+smooth);          rec=(tp+smooth)/(tp+fn+smooth)
        hd=calc_hd95_u(pm.astype(bool),gm.astype(bool))
        cbl=calc_cbl_u(pm.astype(bool),gm.astype(bool))
        unet_all_dice.append(dice); unet_all_iou.append(iou)
        unet_all_pre.append(pre);   unet_all_rec.append(rec)
        unet_all_hd95.append(hd)
        if cbl is not None: unet_all_cbl.append(cbl)

        if idx in SHOW_INDEX:
            img_name = img_names_sorted[idx]
            cx_gt,cy_gt=get_centroid(gm); cx_p,cy_p=get_centroid(pm)
            fig,axes=plt.subplots(1,4,figsize=(20,5))
            fig.suptitle(f'U-Net 2D | {img_name}',fontsize=14,fontweight='bold')
            axes[0].imshow(img_np,cmap='gray'); axes[0].set_title('Ảnh gốc',fontsize=11,fontweight='bold')
            axes[1].imshow(img_np,cmap='gray')
            green=np.zeros((*gm.shape,4),dtype=np.float32); green[gm==1]=[0,1,0,0.3]; axes[1].imshow(green)
            if gm.max()>0: axes[1].contour(gm,[0.5],colors='lime',linewidths=1.5)
            if cx_gt is not None: axes[1].plot(cx_gt,cy_gt,'o',color='lime',ms=8,markeredgecolor='black',label='Tâm GT')
            axes[1].legend(loc='lower right',fontsize=8); axes[1].set_title('Ground Truth',fontsize=11,fontweight='bold')
            axes[2].imshow(img_np,cmap='gray')
            red=np.zeros((*pm.shape,4),dtype=np.float32); red[pm==1]=[1,0,0,0.3]; axes[2].imshow(red)
            if pm.max()>0: axes[2].contour(pm,[0.5],colors='red',linewidths=1.5)
            if cx_p is not None: axes[2].plot(cx_p,cy_p,'o',color='red',ms=8,markeredgecolor='white',label='Tâm Pred')
            axes[2].legend(loc='lower right',fontsize=8); axes[2].set_title('Dự đoán (U-Net)',fontsize=11,fontweight='bold')
            axes[3].imshow(img_np,cmap='gray')
            if gm.max()>0: axes[3].contour(gm,[0.5],colors='lime',linewidths=2)
            if pm.max()>0: axes[3].contour(pm,[0.5],colors='red', linewidths=2,linestyles='--')
            if cx_gt is not None: axes[3].plot(cx_gt,cy_gt,'o',color='lime',ms=8,markeredgecolor='black')
            if cx_p  is not None: axes[3].plot(cx_p, cy_p, 'o',color='red', ms=8,markeredgecolor='white')
            if cx_gt is not None and cx_p is not None:
                axes[3].plot([cx_gt,cx_p],[cy_gt,cy_p],'--',color='yellow',lw=1.5)
            axes[3].set_title(f'Dice:{dice:.3f} IoU:{iou:.3f} HD95:{hd:.1f}px\nCBL:{cbl:.3f} Pre:{pre:.3f} Rec:{rec:.3f}',
                              fontsize=10,fontweight='bold')
            for ax in axes: ax.axis('off')
            plt.tight_layout(); plt.show()

# ── Final report ──────────────────────────────────────────────────────
unet_results = dict(dice=float(np.mean(unet_all_dice)),  iou=float(np.mean(unet_all_iou)),
                    precision=float(np.mean(unet_all_pre)), recall=float(np.mean(unet_all_rec)),
                    hd95=float(np.mean(unet_all_hd95)),  cbl=float(np.mean(unet_all_cbl)))
print('\n'+'='*60)
print('FINAL TEST RESULTS — U-NET 2D | ImgSize=512')
print('='*60)
for k,v in unet_results.items(): print(f'  {k:<12}: {v:.4f}')
print(f'  Total       : {len(unet_all_dice)} samples')
print('='*60)

os.makedirs(f'{UNET_ROOT}/results', exist_ok=True)
with open(f'{UNET_ROOT}/results/unet2d_results.csv','w',newline='') as f:
    w=csv.writer(f)
    w.writerow(['model','dice','iou','precision','recall','hd95','cbl','n_samples'])
    w.writerow(['UNet2D']+[f'{unet_results[k]:.4f}' for k in ['dice','iou','precision','recall','hd95','cbl']]+[len(unet_all_dice)])
print(f'✅ CSV saved')


---
## Tổng hợp & So sánh
---

In [ ]:
# ══════════════════════════════════════════════════════
# TỔNG HỢP — PGA-UNet (zoom_out) vs U-Net 2D (IMG_SIZE=512)
# Tự đọc từ CSV nếu chưa có biến trong bộ nhớ
# ══════════════════════════════════════════════════════
import csv, os, numpy as np
import matplotlib.pyplot as plt

BASE      = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
PGA_ROOT  = f'{BASE}/pga-repo'
UNET_ROOT = f'{BASE}/unet-repo'
KEYS      = ['dice','iou','precision','recall','hd95','cbl']

# ── Tải kết quả: ưu tiên biến bộ nhớ, fallback đọc CSV ──────────────
if 'pga_results' not in dir() or not pga_results:
    with open(f'{PGA_ROOT}/results/pga_unet2d_test_results.csv') as _f:
        pga_results = {}
        for _row in csv.DictReader(_f):
            pga_results[_row['mode']] = {k: float(_row[k]) for k in KEYS}
    print('📂 pga_results  ← CSV')
else:
    print('✅ pga_results  ← memory')

if 'unet_results' not in dir() or not unet_results:
    with open(f'{UNET_ROOT}/results/unet2d_results.csv') as _f:
        _row = next(csv.DictReader(_f))
        unet_results = {k: float(_row[k]) for k in KEYS}
    print('📂 unet_results ← CSV')
else:
    print('✅ unet_results ← memory')

HDRS   = ['Dice↑','IoU↑','Prec↑','Rec↑','HD95↓ (px)','CBL↑']
pga_zo = pga_results['zoom_out']

# ── Bảng tổng hợp ────────────────────────────────────────────────────
bar='═'*80
print(f'\n{bar}')
print('  SO SÁNH — PGA-UNet (zoom_out) vs U-Net 2D (Image-level, IMG_SIZE=512)')
print(f'{bar}')
print(f"  {'Model':<28}"+''.join(f'{h:>9}' for h in HDRS))
print('  '+'-'*77)
print(f"  {'PGA-UNet (zoom_out)':<28}"+''.join(f'{pga_zo[k]:>9.4f}'      for k in KEYS))
print(f"  {'U-Net 2D':<28}"          +''.join(f'{unet_results[k]:>9.4f}' for k in KEYS))
print(f'{bar}')
for k in KEYS:
    delta = pga_zo[k] - unet_results[k]
    suffix = '  (âm = PGA tốt hơn)' if k == 'hd95' else ''
    print(f'  Δ PGA − UNet  [{k.upper():<10}]: {delta:+.4f}{suffix}')

# ── Biểu đồ cột ──────────────────────────────────────────────────────
METRIC_PAIRS = [
    ('dice',      'Dice ↑'),
    ('iou',       'IoU ↑'),
    ('precision', 'Precision ↑'),
    ('recall',    'Recall ↑'),
    ('hd95',      'HD95 ↓ (px)'),
    ('cbl',       'CBL ↑'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
fig.suptitle('PGA-UNet (zoom_out) vs U-Net 2D — So sánh 6 metrics (IMG_SIZE=512)',
             fontsize=14, fontweight='bold', y=1.01)

COLOR_PGA  = '#1565C0'
COLOR_UNET = '#EF5350'
x_pos      = [0, 1]
x_labels   = ['PGA-UNet\n(zoom_out)', 'U-Net 2D\n(no prompt)']

for ax_idx, (metric, label) in enumerate(METRIC_PAIRS):
    ax   = axes[ax_idx]
    vals = [pga_zo[metric], unet_results[metric]]
    clrs = [COLOR_PGA, COLOR_UNET]

    ax.bar(x_pos, vals, 0.5, color=clrs, alpha=0.88, edgecolor='white')
    for xi, val, tc in zip(x_pos, vals, ['#0D47A1','#B71C1C']):
        fmt = f'{val:.0f}' if metric == 'hd95' else f'{val:.4f}'
        ax.text(xi, val + (0.006 if metric != 'hd95' else 0.8), fmt,
                ha='center', va='bottom', fontsize=11, fontweight='bold', color=tc)

    ax.set_title(label, fontsize=12, fontweight='bold', pad=8)
    hi = max(vals)
    ax.set_ylim(0, hi * 1.22 if metric != 'hd95' else hi * 1.30)
    ax.set_xticks(x_pos); ax.set_xticklabels(x_labels, fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

handles = [plt.Rectangle((0,0),1,1,color=COLOR_PGA, alpha=0.88),
           plt.Rectangle((0,0),1,1,color=COLOR_UNET, alpha=0.88)]
fig.legend(handles, ['PGA-UNet (zoom_out)', 'U-Net 2D (no prompt)'],
           loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5,-0.02))

plt.tight_layout()
plt.savefig(f'{BASE}/pga_vs_unet2d_comparison.png', dpi=150, bbox_inches='tight')
from IPython.display import display as _d; _d(fig); plt.close(fig)
print('\n✅ Biểu đồ đã lưu: pga_vs_unet2d_comparison.png')
